# Module 9: Latent Diffusion & Advanced Topics

This module covers how diffusion scales to high-resolution images and the modern architectural directions that power systems like Stable Diffusion, DALL-E 3, and Sora. We move from pixel-space limitations through VAEs, latent diffusion, and into the frontier: DiT and flow matching.

**Learning Objectives**
- Understand why pixel-space diffusion doesn't scale and how latent diffusion solves this
- Implement a convolutional VAE and understand the ELBO loss
- Build a full latent diffusion pipeline: VAE → encode → diffuse in latent space → decode
- Map the Stable Diffusion architecture (VAE + U-Net + text encoder)
- Compare ε-prediction, x₀-prediction, and v-prediction parameterizations
- Understand noise schedule improvements: cosine, offset noise, zero terminal SNR
- Implement a minimal Diffusion Transformer (DiT) with adaLN-Zero
- Implement flow matching — the simpler alternative to DDPM

**Estimated time:** 4–5 hours

**Key references:**
- High-Resolution Image Synthesis with Latent Diffusion Models — Rombach et al. 2022: https://arxiv.org/abs/2112.10752
- Auto-Encoding Variational Bayes — Kingma & Welling 2013: https://arxiv.org/abs/1312.6114
- Scalable Diffusion Models with Transformers (DiT) — Peebles & Xie 2023: https://arxiv.org/abs/2212.09748
- Flow Matching for Generative Modeling — Lipman et al. 2023: https://arxiv.org/abs/2210.02747
- Rectified Flow — Liu et al. 2023: https://arxiv.org/abs/2209.03003
- Elucidating the Design Space (EDM) — Karras et al. 2022: https://arxiv.org/abs/2206.00364

In [ ]:
import sys
import math
import time
from typing import Optional, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, '.')
from utils.schedule import cosine_schedule, linear_schedule, get_schedule
from utils.visualization import show_images, denormalize, plot_loss_curve, set_style
from utils.data import get_mnist_dataloader, get_device

torch.manual_seed(42)
device = get_device()
print(f"Using device: {device}")
set_style()

---
## 9.1 — The Resolution Problem

Pixel-space diffusion works beautifully on small images (28×28 MNIST, 32×32 CIFAR-10). But scaling to real-world resolutions reveals a fundamental bottleneck.

Consider the raw dimensionality:

| Resolution | Channels | Total pixels | Relative cost |
|-----------|----------|-------------|---------------|
| 32×32 | 3 | 3,072 | 1× |
| 64×64 | 3 | 12,288 | 4× |
| 128×128 | 3 | 49,152 | 16× |
| 256×256 | 3 | 196,608 | 64× |
| 512×512 | 3 | 786,432 | 256× |

The U-Net processes feature maps at these spatial resolutions. Self-attention layers scale **quadratically** with spatial size — at 64×64, attention operates over 4,096 tokens; at 256×256 that's 65,536 tokens. Memory and compute explode.

The key insight of latent diffusion: **don't diffuse pixels — diffuse a compressed representation**.

### Worked Example: Profiling the Resolution Bottleneck

In [ ]:
class SimpleConvBlock(nn.Module):
    """A minimal conv block to simulate U-Net processing at various resolutions."""
    def __init__(self, channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1),    # (B, 64, H, W)
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),  # (B, 64, H, W)
            nn.GroupNorm(8, channels),
            nn.SiLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x)  # (B, C, H, W)
        B, C, H, W = h.shape
        flat = h.view(B, C, H * W)        # (B, C, H*W)
        attn = torch.bmm(flat.transpose(1, 2), flat)  # (B, H*W, H*W) — quadratic!
        attn = F.softmax(attn / math.sqrt(C), dim=-1)
        h = torch.bmm(flat, attn.transpose(1, 2)).view(B, C, H, W)
        return h


# Profile memory at different resolutions
resolutions = [32, 64, 128, 256]
results = []

for res in resolutions:
    model = SimpleConvBlock(channels=64)
    x = torch.randn(1, 3, res, res)
    pixels = 3 * res * res
    attn_tokens = res * res
    attn_matrix_size = attn_tokens ** 2

    start = time.time()
    try:
        with torch.no_grad():
            _ = model(x)
        elapsed = time.time() - start
        oom = False
    except RuntimeError:
        elapsed = float('inf')
        oom = True

    results.append({
        'res': res, 'pixels': pixels,
        'attn_tokens': attn_tokens, 'attn_matrix': attn_matrix_size,
        'time_ms': elapsed * 1000, 'oom': oom
    })

print(f"{'Res':>6s} {'Pixels':>10s} {'Attn tokens':>12s} {'Attn matrix':>14s} {'Time (ms)':>10s}")
print("-" * 60)
for r in results:
    t_str = 'OOM' if r['oom'] else f"{r['time_ms']:.1f}"
    print(f"{r['res']:>6d} {r['pixels']:>10,d} {r['attn_tokens']:>12,d} {r['attn_matrix']:>14,d} {t_str:>10s}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
valid = [r for r in results if not r['oom']]
ax1.plot([r['res'] for r in valid], [r['attn_matrix'] for r in valid], 'o-', color='crimson')
ax1.set_xlabel('Resolution'); ax1.set_ylabel('Attention matrix elements')
ax1.set_title('Attention: O(n²) in tokens, O(res⁴) for images'); ax1.set_yscale('log')

ax2.plot([r['res'] for r in valid], [r['time_ms'] for r in valid], 's-', color='steelblue')
ax2.set_xlabel('Resolution'); ax2.set_ylabel('Forward pass time (ms)')
ax2.set_title('Compute time vs resolution')
plt.tight_layout(); plt.show()

print(f"\n256×256 attention matrix: {256**4:,} elements — this is why we need latent diffusion!")

---
## 9.2 — VAE Primer

A **Variational Autoencoder (VAE)** learns to compress images into a low-dimensional latent space and reconstruct them. It has two components:

- **Encoder** $E$: maps image $x$ to a distribution in latent space: $E(x) = (\mu, \log \sigma^2)$
- **Decoder** $D$: maps a latent sample $z$ back to image space: $\hat{x} = D(z)$

### The ELBO Loss

The VAE is trained to maximize the **Evidence Lower Bound (ELBO)**:

$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{Reconstruction}} - \underbrace{D_{\text{KL}}(q(z|x) \| p(z))}_{\text{KL regularization}}$$

In practice:
- **Reconstruction loss**: MSE between input and reconstruction
- **KL divergence**: pushes $q(z|x) = \mathcal{N}(\mu, \sigma^2)$ toward the prior $p(z) = \mathcal{N}(0, I)$

$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

### The Reparameterization Trick

To backpropagate through the sampling step:
$$z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This moves the stochasticity into $\epsilon$, making $z$ a deterministic function of $\mu$, $\sigma$, and $\epsilon$.

### Worked Example: Convolutional VAE on MNIST

In [ ]:
class ConvVAE(nn.Module):
    """Convolutional VAE for 28x28 grayscale images.

    Encoder: 1x28x28 -> 32x14x14 -> 64x7x7 -> flatten -> (mu, log_var) of dim latent_dim
    Decoder: latent_dim -> 64x7x7 -> 32x14x14 -> 1x28x28
    """
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),  # (B, 32, 14, 14)
            nn.GroupNorm(min(32, 32), 32),  # GroupNorm: consistent with diffusion best practices
            nn.SiLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # (B, 64, 7, 7)
            nn.GroupNorm(min(32, 64), 64),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)      # (B, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)   # (B, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim, 64 * 7 * 7)   # (B, 64*7*7)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # (B, 32, 14, 14)
            nn.GroupNorm(min(32, 32), 32),  # GroupNorm: consistent with diffusion best practices
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),   # (B, 1, 28, 28)
            nn.Tanh(),  # Output in [-1, 1] to match data normalization
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode image to latent distribution parameters."""
        h = self.encoder(x)             # (B, 64, 7, 7)
        h = h.view(h.size(0), -1)       # (B, 64*7*7)
        return self.fc_mu(h), self.fc_logvar(h)  # (B, latent_dim), (B, latent_dim)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample z = mu + sigma * epsilon using the reparameterization trick."""
        std = torch.exp(0.5 * logvar)   # (B, latent_dim)
        eps = torch.randn_like(std)     # (B, latent_dim)
        return mu + std * eps           # (B, latent_dim)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector to image."""
        h = self.fc_decode(z)           # (B, 64*7*7)
        h = h.view(-1, 64, 7, 7)       # (B, 64, 7, 7)
        return self.decoder(h)          # (B, 1, 28, 28)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Full forward: encode -> sample -> decode."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(
    x: torch.Tensor, x_recon: torch.Tensor,
    mu: torch.Tensor, logvar: torch.Tensor,
    kl_weight: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute VAE loss = reconstruction + KL divergence."""
    recon = F.mse_loss(x_recon, x, reduction='mean')
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl_weight * kl, recon, kl


# Quick shape check
vae = ConvVAE(latent_dim=32).to(device)
test_x = torch.randn(4, 1, 28, 28, device=device)
recon, mu, logvar = vae(test_x)
print(f"Input shape:  {test_x.shape}")
print(f"Latent shape: {mu.shape}")
print(f"Recon shape:  {recon.shape}")
print(f"Compression:  {28*28}/{mu.shape[1]} = {28*28/mu.shape[1]:.0f}x")

In [ ]:
# Train the VAE on MNIST
torch.manual_seed(42)

vae = ConvVAE(latent_dim=32).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
dataloader = get_mnist_dataloader(batch_size=128)

num_epochs = 10
losses_hist = {'total': [], 'recon': [], 'kl': []}

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for images, _ in dataloader:
        images = images.to(device)  # (B, 1, 28, 28)
        x_recon, mu, logvar = vae(images)
        loss, recon, kl = vae_loss(images, x_recon, mu, logvar, kl_weight=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses_hist['total'].append(avg_loss)
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

print("VAE training complete!")

In [ ]:
# Visualize VAE reconstructions and latent interpolation
vae.eval()
test_batch, _ = next(iter(dataloader))
test_batch = test_batch[:8].to(device)

with torch.no_grad():
    recon, _, _ = vae(test_batch)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(denormalize(test_batch[i]).cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(denormalize(recon[i]).cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Recon', fontsize=12)
plt.suptitle('VAE Reconstructions', fontsize=14)
plt.tight_layout(); plt.show()

# Latent space interpolation
with torch.no_grad():
    mu1, _ = vae.encode(test_batch[0:1])
    mu2, _ = vae.encode(test_batch[4:5])
    n_steps = 10
    alphas_interp = torch.linspace(0, 1, n_steps, device=device)
    interp_z = torch.stack([mu1 * (1 - a) + mu2 * a for a in alphas_interp]).squeeze(1)
    interp_imgs = vae.decode(interp_z)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
for i in range(n_steps):
    axes[i].imshow(denormalize(interp_imgs[i]).cpu().squeeze(), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'{alphas_interp[i]:.1f}')
plt.suptitle('Latent Space Interpolation', fontsize=14)
plt.tight_layout(); plt.show()

### Exercise 9.1: Explore VAE Latent Dimension

Train two VAEs with different latent dimensions (e.g., `latent_dim=8` and `latent_dim=64`) for 5 epochs each. Compare:
1. Reconstruction quality
2. Samples from the prior $z \sim \mathcal{N}(0, I)$ decoded to images

In [ ]:
# Exercise 9.1 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for row, ldim in enumerate([8, 32, 64]):
    vae_test = ConvVAE(latent_dim=ldim).to(device)
    opt = torch.optim.Adam(vae_test.parameters(), lr=1e-3)
    for epoch in range(5):
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            recon, mu, logvar = vae_test(imgs)
            loss, _, _ = vae_loss(imgs, recon, mu, logvar, kl_weight=0.5)
            opt.zero_grad(); loss.backward(); opt.step()
    vae_test.eval()
    with torch.no_grad():
        z_random = torch.randn(8, ldim, device=device)
        samples = vae_test.decode(z_random)
    for i in range(8):
        axes[row, i].imshow(denormalize(samples[i]).cpu().squeeze(), cmap='gray')
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(f'dim={ldim}', fontsize=12)
plt.suptitle('Random Samples from Prior at Different Latent Dims', fontsize=14)
plt.tight_layout(); plt.show()
print("Lower dim -> blurrier but smoother latent space. Higher dim -> sharper but harder to sample.")

---
## 9.3 — Latent Diffusion: Compress → Diffuse → Decode

The key idea of **Latent Diffusion Models (LDMs)**:

1. **Train a VAE** on images to learn encoder $E$ and decoder $D$
2. **Encode** all training images to latent space: $z_0 = E(x)$
3. **Train a diffusion model** on $z_0$ instead of $x$ — much smaller!
4. **At inference**: sample $z_0$ from diffusion, then $\hat{x} = D(z_0)$

$$\text{Image } x \xrightarrow{\text{Encoder}} z_0 \xrightarrow{\text{Diffusion}} z_T \sim \mathcal{N}(0, I)$$

$$\mathcal{N}(0, I) \sim z_T \xrightarrow{\text{Denoise}} z_0 \xrightarrow{\text{Decoder}} \hat{x}$$

### Why this works

- **Lower dimensionality**: 784 pixels (28×28) → 32 latent dims = **24× compression**
- **Perceptual compression**: the VAE discards imperceptible detail
- **Semantic focus**: the diffusion model learns high-level structure, not pixel noise

In Stable Diffusion: 512×512×3 → 64×64×4 = **48× compression**.

### Worked Example: Latent Diffusion on MNIST

In [ ]:
# Step 1: Encode all MNIST training images to latent space
vae.eval()
all_latents = []
with torch.no_grad():
    for images, _ in tqdm(dataloader, desc="Encoding MNIST"):
        images = images.to(device)
        mu, _ = vae.encode(images)  # (B, 32) — use mean, no sampling noise
        all_latents.append(mu.cpu())

latent_dataset = torch.cat(all_latents, dim=0)  # (60000, 32)
print(f"Latent dataset shape: {latent_dataset.shape}")
print(f"Latent stats: mean={latent_dataset.mean():.3f}, std={latent_dataset.std():.3f}")

# Normalize latents to roughly unit variance for diffusion
latent_mean = latent_dataset.mean(dim=0, keepdim=True)  # (1, 32)
latent_std = latent_dataset.std(dim=0, keepdim=True)    # (1, 32)
latent_dataset_norm = (latent_dataset - latent_mean) / (latent_std + 1e-6)
print(f"Normalized stats: mean={latent_dataset_norm.mean():.3f}, std={latent_dataset_norm.std():.3f}")

In [ ]:
class LatentDiffusionMLP(nn.Module):
    """Simple MLP for diffusion on 1D latent codes.

    Since our latents are flat vectors (not spatial), we use an MLP
    instead of a U-Net. Predicts noise given (z_t, t).
    """
    def __init__(self, latent_dim: int = 32, hidden_dim: int = 256, time_dim: int = 64):
        super().__init__()
        # A single nn.Linear(1, time_dim) is insufficient for time conditioning —
        # the model needs nonlinear features of t to distinguish noise levels.
        # A small MLP (or sinusoidal embedding) gives much better results.
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),       # (B, time_dim)
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_dim, hidden_dim),  # (B, hidden_dim)
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),  # (B, latent_dim)
        )

    def forward(self, z_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict noise given noisy latent and timestep.

        Args:
            z_t: (B, latent_dim) noisy latent
            t: (B,) or (B, 1) timestep in [0, 1]
        """
        if t.dim() == 1:
            t = t.unsqueeze(1)
        t_emb = self.time_embed(t)           # (B, time_dim)
        x = torch.cat([z_t, t_emb], dim=1)  # (B, latent_dim + time_dim)
        return self.net(x)                   # (B, latent_dim)"

In [ ]:
# Step 2: Train diffusion model on latent codes
torch.manual_seed(42)

T = 1000
schedule = cosine_schedule(T)
sqrt_alpha_bar = schedule['sqrt_alphas_cumprod'].to(device)
sqrt_one_minus_alpha_bar = schedule['sqrt_one_minus_alphas_cumprod'].to(device)

latent_model = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(latent_model.parameters(), lr=1e-3)

latent_loader = DataLoader(TensorDataset(latent_dataset_norm), batch_size=256, shuffle=True)

num_epochs = 30
loss_history = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for (z_0,) in latent_loader:
        z_0 = z_0.to(device)  # (B, 32)
        B = z_0.shape[0]
        t_idx = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(z_0)  # (B, 32)
        z_t = sqrt_alpha_bar[t_idx].unsqueeze(1) * z_0 + \
              sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise  # (B, 32)
        t_norm = t_idx.float() / T
        noise_pred = latent_model(z_t, t_norm)
        loss = F.mse_loss(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(latent_loader)
    loss_history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

plot_loss_curve(loss_history, title='Latent Diffusion Training Loss')

In [ ]:
# Step 3: Sample from latent diffusion -> decode to images
@torch.no_grad()
def sample_latent_diffusion(
    model: LatentDiffusionMLP,
    schedule: Dict[str, torch.Tensor],
    n_samples: int = 16,
    latent_dim: int = 32,
    device: torch.device = torch.device('cpu'),
) -> torch.Tensor:
    """DDPM sampling in latent space."""
    T = len(schedule['betas'])
    betas = schedule['betas'].to(device)
    alphas = schedule['alphas'].to(device)
    alphas_cumprod = schedule['alphas_cumprod'].to(device)
    posterior_var = schedule['posterior_variance'].to(device)

    z = torch.randn(n_samples, latent_dim, device=device)  # (N, 32)
    for t in reversed(range(T)):
        t_norm = torch.full((n_samples,), t / T, device=device)
        eps_pred = model(z, t_norm)
        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]
        mean = (1 / alpha_t.sqrt()) * (z - (beta_t / (1 - alpha_bar_t).sqrt()) * eps_pred)
        if t > 0:
            z = mean + posterior_var[t].sqrt() * torch.randn_like(z)
        else:
            z = mean
    return z

latent_model.eval()
vae.eval()
sampled_z = sample_latent_diffusion(latent_model, schedule, n_samples=16, latent_dim=32, device=device)
sampled_z = sampled_z * latent_std.to(device) + latent_mean.to(device)  # denormalize

with torch.no_grad():
    generated = vae.decode(sampled_z)  # (16, 1, 28, 28)

show_images(denormalize(generated), nrow=8, title='Latent Diffusion Samples (VAE + MLP Diffusion)')
print("These images were generated by diffusing in a 32-dim latent space, not pixel space!")

### Exercise 9.2: Compare Pixel-Space vs Latent Diffusion

Time how long it takes to train a pixel-space MLP diffusion model on flattened MNIST (784 dims) vs a latent-space MLP (32 dims) for 10 epochs each. Compare training time and final loss.

In [ ]:
# Exercise 9.2 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Pixel-space: flatten MNIST
all_pixels = []
for images, _ in dataloader:
    all_pixels.append(images.view(images.size(0), -1))
pixel_dataset = torch.cat(all_pixels, dim=0)  # (60000, 784)
pixel_loader = DataLoader(TensorDataset(pixel_dataset), batch_size=256, shuffle=True)

for name, dim, loader in [('Pixel (784d)', 784, pixel_loader), ('Latent (32d)', 32, latent_loader)]:
    model = LatentDiffusionMLP(latent_dim=dim, hidden_dim=min(dim * 4, 512)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    start = time.time()
    final_loss = 0
    for epoch in range(10):
        for (batch,) in loader:
            batch = batch.to(device)
            B = batch.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(batch)
            noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                    sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
            pred = model(noisy, t_idx.float() / T)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
            final_loss = loss.item()
    elapsed = time.time() - start
    print(f"{name}: {elapsed:.1f}s for 10 epochs, final loss={final_loss:.4f}")

---
## 9.4 — Stable Diffusion Architecture Overview

Stable Diffusion is a latent diffusion model with three core components:

```
 STABLE DIFFUSION PIPELINE

 "a photo of a cat"
        |
        v
 +--------------+
 | Text Encoder  |  CLIP or T5: text -> embeddings (77, 768)
 | (frozen)      |
 +------+-------+
        | cross-attention
        v
 +--------------+     z_T ~ N(0,I)
 |   U-Net       |<-- (64x64x4 latent)
 | (trainable)   |  Iterative denoising with CFG
 |               |  z_T -> z_{T-1} -> ... -> z_0
 +------+-------+
        |
        v
 +--------------+
 | VAE Decoder   |  z_0 (64x64x4) -> image (512x512x3)
 | (frozen)      |
 +--------------+
```

### Component Details

| Component | Input → Output | Role |
|-----------|---------------|------|
| **VAE Encoder** | 512×512×3 → 64×64×4 | Compress image to latent (training only) |
| **VAE Decoder** | 64×64×4 → 512×512×3 | Decode latent to image (inference) |
| **U-Net** | 64×64×4 → 64×64×4 | Denoise latent, conditioned on text + time |
| **Text Encoder** | String → (77, 768) | Convert text to embedding sequence |

### The Inference Pipeline

1. **Encode prompt**: text → CLIP → embeddings
2. **Sample noise**: $z_T \sim \mathcal{N}(0, I)$ at 64×64×4
3. **Iterative denoising** (with CFG from Module 8):
   $\hat{\epsilon}_{\text{guided}} = \epsilon_\theta(z_t, t, \varnothing) + s \cdot \big(\epsilon_\theta(z_t, t, c_{\text{text}}) - \epsilon_\theta(z_t, t, \varnothing)\big)$
4. **Decode**: $\hat{x} = D(z_0)$

### Evolution

| Version | Text Encoder | Architecture | Resolution |
|---------|-------------|-------------|------------|
| SD 1.5 | CLIP ViT-L/14 | U-Net 860M | 512×512 |
| SD 2.1 | OpenCLIP ViT-H | U-Net 860M + v-pred | 768×768 |
| SDXL | Dual CLIP | U-Net 2.6B | 1024×1024 |
| SD 3 | T5-XXL + 2×CLIP | DiT (MMDiT) + flow matching | 1024×1024 |

We won't implement the full pipeline (it requires pretrained weights), but you should be able to describe every component and how they connect.

---
## 9.5 — v-Prediction vs ε-Prediction vs x₀-Prediction

The diffusion model must predict *something* given $(x_t, t)$. Three standard parameterizations:

### 1. ε-Prediction (DDPM standard)
Predict the noise: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
$$\mathcal{L} = \|\epsilon - \epsilon_\theta(x_t, t)\|^2$$

### 2. x₀-Prediction
Predict the clean data: $\hat{x}_0 = f_\theta(x_t, t)$
$$\mathcal{L} = \|x_0 - f_\theta(x_t, t)\|^2$$

### 3. v-Prediction (Imagen, SD 2.x)
Predict the "velocity": $v = \sqrt{\bar{\alpha}_t} \cdot \epsilon - \sqrt{1 - \bar{\alpha}_t} \cdot x_0$
$$\mathcal{L} = \|v - v_\theta(x_t, t)\|^2$$

### Conversion Formulas

All three are interconvertible given $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$:

| From | To $x_0$ | To $\epsilon$ |
|------|----------|---------------|
| $\hat{\epsilon}$ | $x_0 = (x_t - \sqrt{1-\bar{\alpha}_t}\hat{\epsilon})/\sqrt{\bar{\alpha}_t}$ | — |
| $\hat{x}_0$ | — | $\epsilon = (x_t - \sqrt{\bar{\alpha}_t}\hat{x}_0)/\sqrt{1-\bar{\alpha}_t}$ |
| $\hat{v}$ | $x_0 = \sqrt{\bar{\alpha}_t} x_t - \sqrt{1-\bar{\alpha}_t} \hat{v}$ | $\epsilon = \sqrt{1-\bar{\alpha}_t} x_t + \sqrt{\bar{\alpha}_t} \hat{v}$ |

**Why v-prediction?**
- ε-prediction has high variance near $t=0$ (low noise)
- x₀-prediction has high variance near $t=T$ (high noise)
- v-prediction interpolates smoothly — balanced across all timesteps

### Worked Example: Conversion Functions and Equivalence

In [ ]:
def compute_v_target(x_0, epsilon, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    """v = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0"""
    return sqrt_alpha_bar * epsilon - sqrt_one_minus_alpha_bar * x_0

def eps_to_x0(x_t, eps, sab, somab):
    """Convert eps-prediction to x0-prediction."""
    return (x_t - somab * eps) / sab

def x0_to_eps(x_t, x0, sab, somab):
    """Convert x0-prediction to eps-prediction."""
    return (x_t - sab * x0) / somab

def v_to_x0(x_t, v, sab, somab):
    """Convert v-prediction to x0-prediction."""
    return sab * x_t - somab * v

def v_to_eps(x_t, v, sab, somab):
    """Convert v-prediction to eps-prediction."""
    return somab * x_t + sab * v


# Demonstrate equivalence
torch.manual_seed(42)
x_0 = torch.randn(4, 32)
eps = torch.randn(4, 32)
t = 500
sab = schedule['sqrt_alphas_cumprod'][t]
somab = schedule['sqrt_one_minus_alphas_cumprod'][t]
x_t = sab * x_0 + somab * eps
v = compute_v_target(x_0, eps, sab, somab)

# Verify round-trip conversions
print(f"x0 from eps: max error = {(eps_to_x0(x_t, eps, sab, somab) - x_0).abs().max():.2e}")
print(f"x0 from v:   max error = {(v_to_x0(x_t, v, sab, somab) - x_0).abs().max():.2e}")
print(f"eps from v:  max error = {(v_to_eps(x_t, v, sab, somab) - eps).abs().max():.2e}")
print("\nAll three parameterizations are mathematically equivalent!")

In [ ]:
# Compare loss magnitude across timesteps for each parameterization
torch.manual_seed(42)
timesteps = torch.linspace(0, T-1, 50).long()
x_0_sample = torch.randn(64, 32)

eps_losses, x0_losses, v_losses = [], [], []
for t_val in timesteps:
    sab = schedule['sqrt_alphas_cumprod'][t_val]
    somab = schedule['sqrt_one_minus_alphas_cumprod'][t_val]
    eps = torch.randn_like(x_0_sample)
    x_t = sab * x_0_sample + somab * eps
    v_target = compute_v_target(x_0_sample, eps, sab, somab)
    pred_error = 0.1 * torch.randn_like(x_0_sample)

    eps_losses.append((eps_to_x0(x_t, eps + pred_error, sab, somab) - x_0_sample).pow(2).mean().item())
    x0_losses.append(pred_error.pow(2).mean().item())
    v_losses.append((v_to_x0(x_t, v_target + pred_error, sab, somab) - x_0_sample).pow(2).mean().item())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(timesteps.numpy(), eps_losses, label='eps-prediction', color='crimson')
ax.plot(timesteps.numpy(), x0_losses, label='x0-prediction', color='steelblue')
ax.plot(timesteps.numpy(), v_losses, label='v-prediction', color='forestgreen')
ax.set_xlabel('Timestep t'); ax.set_ylabel('x0 recovery MSE (from same pred error)')
ax.set_title('Prediction Error Impact Across Timesteps')
ax.legend(); ax.set_yscale('log')
plt.tight_layout(); plt.show()
print("eps-prediction: unstable near t=0 (divides by small sqrt_alpha_bar)")
print("x0-prediction: constant error (directly predicts x0)")
print("v-prediction: balanced across all timesteps")

### Exercise 9.3: Train All Three Prediction Modes

Modify the `LatentDiffusionMLP` training loop to support all three parameterizations. Train for 15 epochs each and compare final losses.

In [ ]:
# Exercise 9.3 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)
results_pred = {}
for mode in ['eps', 'x0', 'v']:
    model = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(15):
        epoch_loss = 0
        for (z_0,) in latent_loader:
            z_0 = z_0.to(device)
            B = z_0.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(z_0)
            sab_t = sqrt_alpha_bar[t_idx].unsqueeze(1)
            somab_t = sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1)
            z_t = sab_t * z_0 + somab_t * noise
            pred = model(z_t, t_idx.float() / T)
            if mode == 'eps':
                target = noise
            elif mode == 'x0':
                target = z_0
            else:
                target = sab_t * noise - somab_t * z_0
            loss = F.mse_loss(pred, target)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(latent_loader))
    results_pred[mode] = losses
    print(f"{mode}-prediction: final loss = {losses[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
for mode, losses in results_pred.items():
    ax.plot(losses, label=f'{mode}-prediction')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss by Prediction Parameterization')
ax.legend(); plt.tight_layout(); plt.show()

---
## 9.6 — Noise Schedule Improvements

### Cosine Schedule (Improved DDPM)
Smoother SNR transition than linear:
$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right)^2$$

### Offset Noise
Standard Gaussian noise has zero mean — the model can never generate uniform brightness. **Offset noise** adds a per-channel bias:
$$\epsilon = \epsilon_{\text{standard}} + \delta \cdot \epsilon_{\text{offset}}, \quad \epsilon_{\text{offset}} \in \mathbb{R}^{B \times C \times 1 \times 1}$$

### Zero Terminal SNR
Many schedules don't reach $\bar{\alpha}_T = 0$, creating a train-inference mismatch. Fix:
$$\bar{\alpha}_t' = \frac{\bar{\alpha}_t - \bar{\alpha}_T}{1 - \bar{\alpha}_T}$$

### Log-SNR Linear Schedule
Linear in log-SNR space = equal difficulty across timesteps.

### Worked Example: Schedule Comparison

In [ ]:
T_sched = 1000
lin_sched = linear_schedule(T_sched)
cos_sched = cosine_schedule(T_sched)

# Log-SNR linear schedule
def log_snr_linear_schedule(T, snr_min=0.001, snr_max=1000.0):
    """Schedule linear in log-SNR space."""
    log_snr = torch.linspace(math.log(snr_max), math.log(snr_min), T)
    snr = log_snr.exp()
    return snr / (1 + snr)  # SNR = a_bar/(1-a_bar) -> a_bar = SNR/(1+SNR)

log_snr_abar = log_snr_linear_schedule(T_sched)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t_range = np.arange(T_sched)

axes[0].plot(t_range, lin_sched['alphas_cumprod'].numpy(), label='Linear', alpha=0.8)
axes[0].plot(t_range, cos_sched['alphas_cumprod'].numpy(), label='Cosine', alpha=0.8)
axes[0].plot(t_range, log_snr_abar.numpy(), label='Log-SNR linear', alpha=0.8)
axes[0].set_xlabel('Timestep'); axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Cumulative Signal Retention'); axes[0].legend()

for sched, label in [(lin_sched, 'Linear'), (cos_sched, 'Cosine')]:
    abar = sched['alphas_cumprod'].clamp(min=1e-8)
    snr = abar / (1 - abar).clamp(min=1e-8)
    axes[1].plot(t_range, snr.log().numpy(), label=label, alpha=0.8)
snr_ls = log_snr_abar / (1 - log_snr_abar).clamp(min=1e-8)
axes[1].plot(t_range, snr_ls.log().numpy(), label='Log-SNR linear', alpha=0.8)
axes[1].set_xlabel('Timestep'); axes[1].set_ylabel('log SNR')
axes[1].set_title('Log Signal-to-Noise Ratio'); axes[1].legend()

axes[2].bar(['Linear', 'Cosine', 'Log-SNR'],
    [lin_sched['alphas_cumprod'][-1].item(), cos_sched['alphas_cumprod'][-1].item(), log_snr_abar[-1].item()],
    color=['steelblue', 'coral', 'forestgreen'])
axes[2].set_ylabel('alpha_bar_T (should be ~0)')
axes[2].set_title('Terminal SNR (lower = better)'); axes[2].set_yscale('log')
plt.tight_layout(); plt.show()

In [ ]:
# Offset noise demonstration
torch.manual_seed(42)
dark_img = torch.full((1, 1, 28, 28), -0.8)
bright_img = torch.full((1, 1, 28, 28), 0.8)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, (img, label) in enumerate([(dark_img, 'Dark'), (bright_img, 'Bright')]):
    std_noise = torch.randn_like(img)
    noisy_std = 0.5 * img + 0.5 * std_noise

    offset = 0.1 * torch.randn(1, 1, 1, 1)
    offset_noise = std_noise + offset
    noisy_offset = 0.5 * img + 0.5 * offset_noise

    axes[row, 0].imshow(denormalize(img[0]).squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'{label} original')
    axes[row, 1].imshow(denormalize(noisy_std[0]).squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'+ Standard noise')
    axes[row, 2].imshow(denormalize(noisy_offset[0]).squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title(f'+ Offset noise')
    axes[row, 3].hist(std_noise.flatten().numpy(), bins=40, alpha=0.6, label='Std', color='steelblue')
    axes[row, 3].hist(offset_noise.flatten().numpy(), bins=40, alpha=0.6, label='Offset', color='coral')
    axes[row, 3].legend(fontsize=8); axes[row, 3].set_title('Noise distribution')
for ax in axes[:, :3].flat:
    ax.axis('off')
plt.suptitle('Offset Noise enables generating very dark/bright images', fontsize=14)
plt.tight_layout(); plt.show()
print("Standard noise: mean~0 at every pixel -> can't generate uniform brightness.")
print("Offset noise: shifts the mean -> model learns global brightness distribution.")

### Exercise 9.4: Zero Terminal SNR Schedule

Implement a rescaled cosine schedule that enforces zero terminal SNR:
$$\bar{\alpha}_t' = \frac{\bar{\alpha}_t - \bar{\alpha}_T}{1 - \bar{\alpha}_T}$$

Plot original vs rescaled and verify $\bar{\alpha}_T' = 0$.

In [ ]:
# Exercise 9.4 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
cos_abar = cos_sched['alphas_cumprod']
abar_T = cos_abar[-1]
cos_abar_rescaled = ((cos_abar - abar_T) / (1 - abar_T)).clamp(min=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cos_abar.numpy(), label=f'Cosine (alpha_bar_T = {abar_T:.4f})', alpha=0.8)
ax.plot(cos_abar_rescaled.numpy(), label=f'Rescaled (alpha_bar_T = {cos_abar_rescaled[-1]:.4f})', alpha=0.8)
ax.set_xlabel('Timestep'); ax.set_ylabel('alpha_bar_t')
ax.set_title('Zero Terminal SNR Rescaling'); ax.legend()
plt.tight_layout(); plt.show()
print(f"Original terminal:  alpha_bar_T = {abar_T:.6f}")
print(f"Rescaled terminal:  alpha_bar_T = {cos_abar_rescaled[-1]:.6f}")

---
## 9.7 — DiT: Diffusion Transformer

The **Diffusion Transformer (DiT)** replaces the U-Net with a vision transformer. Used in Sora, Stable Diffusion 3.

### Architecture
1. **Patchify**: split image into patches, project to embeddings
2. **Positional encoding**: add learned position embeddings
3. **Transformer blocks**: self-attention + FFN with **adaLN-Zero** conditioning
4. **Unpatchify**: reshape back to spatial grid

### adaLN-Zero: Adaptive Layer Norm
$$\text{adaLN}(h, c) = \gamma(c) \cdot \text{LayerNorm}(h) + \beta(c)$$

The "Zero" variant also predicts a gate $\alpha(c)$ initialized to zero, so each block starts as identity.

### Why DiT?
- **Scaling**: transformers scale more predictably with compute
- **Simplicity**: no skip connections, no encoder-decoder asymmetry
- **Flexibility**: handles variable-length sequences

### Worked Example: Minimal DiT Implementation

In [ ]:
class PatchEmbed(nn.Module):
    """Split image into patches and project to embedding dimension."""
    def __init__(self, img_size=28, patch_size=7, in_channels=1, embed_dim=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """(B, C, H, W) -> (B, num_patches, embed_dim)"""
        x = self.proj(x)                      # (B, embed_dim, H/P, W/P)
        return x.flatten(2).transpose(1, 2)   # (B, num_patches, embed_dim)


class AdaLNZero(nn.Module):
    """Adaptive Layer Norm with Zero-initialized gate."""
    def __init__(self, embed_dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.proj = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 3 * embed_dim))
        nn.init.zeros_(self.proj[1].weight)
        nn.init.zeros_(self.proj[1].bias)

    def forward(self, x, cond):
        """Returns (normalized_x with scale+shift, gate)."""
        shift, scale, gate = self.proj(cond).chunk(3, dim=-1)  # Each (B, D)
        x_norm = self.norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        return x_norm, gate.unsqueeze(1)  # (B, N, D), (B, 1, D)


class DiTBlock(nn.Module):
    """Single DiT transformer block with adaLN-Zero."""
    def __init__(self, embed_dim=128, num_heads=4, cond_dim=64, mlp_ratio=4.0):
        super().__init__()
        self.adaln_attn = AdaLNZero(embed_dim, cond_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.adaln_mlp = AdaLNZero(embed_dim, cond_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(embed_dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, embed_dim))

    def forward(self, x, cond):
        """(B, N, D), (B, cond_dim) -> (B, N, D)"""
        x_norm, gate_a = self.adaln_attn(x, cond)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + gate_a * attn_out
        x_norm, gate_m = self.adaln_mlp(x, cond)
        x = x + gate_m * self.mlp(x_norm)
        return x


class DiT(nn.Module):
    """Minimal Diffusion Transformer for 28x28 grayscale images."""
    def __init__(self, img_size=28, patch_size=7, in_channels=1,
                 embed_dim=128, depth=4, num_heads=4, num_classes=10, time_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        cond_dim = time_dim

        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.normal_(self.pos_embed, std=0.02)

        self.time_embed = nn.Sequential(nn.Linear(1, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.class_embed = nn.Embedding(num_classes + 1, time_dim)  # +1 for unconditional

        self.blocks = nn.ModuleList([DiTBlock(embed_dim, num_heads, cond_dim) for _ in range(depth)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.final_proj = nn.Linear(embed_dim, patch_size * patch_size * in_channels)

    def unpatchify(self, x):
        """(B, N, P*P*C) -> (B, C, H, W)"""
        P = self.patch_size
        H = W = int(self.num_patches ** 0.5)
        x = x.view(-1, H, W, P, P, 1)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous()
        return x.view(-1, 1, H * P, W * P)

    def forward(self, x, t, y=None):
        """(B, C, H, W), (B,), optional (B,) -> (B, C, H, W)"""
        cond = self.time_embed(t.unsqueeze(1))
        if y is not None:
            cond = cond + self.class_embed(y)
        h = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            h = block(h, cond)
        h = self.final_proj(self.final_norm(h))
        return self.unpatchify(h)


# Shape verification
dit = DiT(embed_dim=128, depth=4).to(device)
test_x = torch.randn(2, 1, 28, 28, device=device)
test_t = torch.rand(2, device=device)
test_y = torch.tensor([3, 7], device=device)
out = dit(test_x, test_t, test_y)
print(f"Input:  {test_x.shape}")
print(f"Output: {out.shape}")
print(f"DiT params: {sum(p.numel() for p in dit.parameters()):,}")

### Exercise 9.5: Train DiT on MNIST

Train the DiT above on MNIST for 10 epochs using cosine schedule and DDPM training. Use class conditioning with 10% label dropout for CFG (use class 10 as null token).

In [ ]:
# Exercise 9.5 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

dit_model = DiT(embed_dim=128, depth=4, num_heads=4).to(device)
optimizer = torch.optim.Adam(dit_model.parameters(), lr=1e-4)
dataloader = get_mnist_dataloader(batch_size=64)

T_dit = 1000
sched_dit = cosine_schedule(T_dit)
sqrt_abar = sched_dit['sqrt_alphas_cumprod'].to(device)
sqrt_omabar = sched_dit['sqrt_one_minus_alphas_cumprod'].to(device)
null_class = 10
p_uncond = 0.1

loss_history_dit = []
for epoch in range(10):
    epoch_loss = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        B = images.shape[0]
        # Label dropout
        drop_mask = torch.rand(B, device=device) < p_uncond
        labels = torch.where(drop_mask, torch.full_like(labels, null_class), labels)
        # Forward diffusion
        t_idx = torch.randint(0, T_dit, (B,), device=device)
        noise = torch.randn_like(images)
        x_t = sqrt_abar[t_idx, None, None, None] * images + sqrt_omabar[t_idx, None, None, None] * noise
        noise_pred = dit_model(x_t, t_idx.float() / T_dit, labels)
        loss = F.mse_loss(noise_pred, noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(dataloader)
    loss_history_dit.append(avg)
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/10 | Loss: {avg:.4f}")

plot_loss_curve(loss_history_dit, title='DiT Training Loss on MNIST')
print("DiT training complete! Transformers work for diffusion.")

---
## 9.8 — Rectified Flows / Flow Matching

**Flow matching** is a simpler framework that learns a **vector field** transporting samples from noise to data along straight paths.

### Key Idea

Define a path from data $x_0$ (at $t=0$) to noise $x_1 \sim \mathcal{N}(0, I)$ (at $t=1$):
$$x_t = (1 - t) \cdot x_0 + t \cdot x_1$$

The velocity along this path: $dx_t/dt = x_1 - x_0$

Train $v_\theta(x_t, t)$ to predict this velocity:
$$\mathcal{L}_{\text{FM}} = \mathbb{E}_{t, x_0, x_1} \left[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

### Sampling

Start from $x_1 \sim \mathcal{N}(0, I)$ and integrate backward:
$$x_{t-\Delta t} = x_t - \Delta t \cdot v_\theta(x_t, t)$$

### ⚠️ Time Convention Note

This notebook uses the convention **t=0 is data, t=1 is noise**. Some references (e.g., certain DDPM formulations) use the opposite convention (t=0 is noise, t=1 is data). When reading papers or other implementations, always check which convention is used — getting this backwards will silently produce wrong results.

### Why Flow Matching?
1. **Simpler objective**: no noise schedule, no $\bar{\alpha}_t$
2. **Straighter paths**: fewer ODE steps needed
3. **DDPM is a special case**: with a specific non-linear path
4. Used in: Stable Diffusion 3, modern architectures

### Worked Example: Flow Matching on 2D Point Clouds

In [ ]:
# Generate 2D target distribution (two moons)
def make_moons(n=1000, noise=0.05):
    """Generate two-moons dataset."""
    t = torch.linspace(0, math.pi, n // 2)
    upper = torch.stack([torch.cos(t), torch.sin(t)], dim=1)
    lower = torch.stack([1 - torch.cos(t), 1 - torch.sin(t) - 0.5], dim=1)
    data = torch.cat([upper, lower], dim=0)
    data += noise * torch.randn_like(data)
    return data


class FlowMatchingMLP(nn.Module):
    """Simple MLP that predicts velocity v(x_t, t)."""
    def __init__(self, dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x, t):
        """(B, dim), (B, 1) -> (B, dim)"""
        if t.dim() == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat([x, t], dim=1))


# Train flow matching on 2D moons
torch.manual_seed(42)
target_data = make_moons(n=2000, noise=0.05).to(device)
fm_model = FlowMatchingMLP(dim=2, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(fm_model.parameters(), lr=1e-3)

losses_fm = []
for step in range(5000):
    idx = torch.randint(0, len(target_data), (256,))
    x_0 = target_data[idx]                              # (B, 2) data
    x_1 = torch.randn(256, 2, device=device)            # (B, 2) noise
    t = torch.rand(256, 1, device=device)                # (B, 1) in [0, 1]
    x_t = (1 - t) * x_0 + t * x_1                       # (B, 2) interpolation
    target_v = x_1 - x_0                                 # (B, 2) velocity
    pred_v = fm_model(x_t, t)                            # (B, 2)
    loss = F.mse_loss(pred_v, target_v)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 500 == 0:
        losses_fm.append(loss.item())

print(f"Final loss: {losses_fm[-1]:.4f}")

In [ ]:
# Sample from trained flow matching model
@torch.no_grad()
def flow_matching_sample(model, n_samples=500, dim=2, n_steps=50, device=torch.device('cpu')):
    """Sample by integrating learned vector field from t=1 (noise) to t=0 (data)."""
    x = torch.randn(n_samples, dim, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((n_samples, 1), 1 - i * dt, device=device)
        x = x - dt * model(x, t)  # Euler step backward
    return x

fm_model.eval()
samples = flow_matching_sample(fm_model, n_samples=1000, n_steps=50, device=device)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(target_data[:, 0].cpu(), target_data[:, 1].cpu(), s=2, alpha=0.5, c='steelblue')
axes[0].set_title('Target Distribution'); axes[0].set_xlim(-2, 3); axes[0].set_ylim(-1.5, 2)

axes[1].scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=2, alpha=0.5, c='coral')
axes[1].set_title('Flow Matching Samples'); axes[1].set_xlim(-2, 3); axes[1].set_ylim(-1.5, 2)

# Quiver plot of learned vector field
xx, yy = np.meshgrid(np.linspace(-2, 3, 20), np.linspace(-1.5, 2, 15))
grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    vecs = fm_model(grid, torch.full((len(grid), 1), 0.5, device=device)).cpu().numpy()
axes[2].quiver(xx.ravel(), yy.ravel(), -vecs[:, 0], -vecs[:, 1], scale=30, alpha=0.7, color='forestgreen')
axes[2].set_title('Learned Vector Field (t=0.5)'); axes[2].set_xlim(-2, 3); axes[2].set_ylim(-1.5, 2)

plt.suptitle('Flow Matching: 2D Two-Moons', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Compare: sampling quality vs number of steps
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for i, n_steps in enumerate([5, 10, 25, 50]):
    s = flow_matching_sample(fm_model, n_samples=500, n_steps=n_steps, device=device)
    axes[i].scatter(s[:, 0].cpu(), s[:, 1].cpu(), s=2, alpha=0.5, c='coral')
    axes[i].scatter(target_data[:500, 0].cpu(), target_data[:500, 1].cpu(), s=2, alpha=0.2, c='steelblue')
    axes[i].set_title(f'{n_steps} steps'); axes[i].set_xlim(-2, 3); axes[i].set_ylim(-1.5, 2)
plt.suptitle('Flow Matching: Sample Quality vs Number of Steps', fontsize=14)
plt.tight_layout(); plt.show()
print("Straight paths -> good samples even with very few steps!")

### Exercise 9.6: Flow Matching on MNIST Latents

Apply flow matching to the MNIST latent codes from Section 9.3:
1. Train a `FlowMatchingMLP` with `dim=32` for 30 epochs on `latent_dataset_norm`
2. Sample latents using ODE solver, decode through VAE
3. Compare with DDPM-based latent diffusion from 9.3

In [ ]:
# Exercise 9.6 — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

fm_latent = FlowMatchingMLP(dim=32, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(fm_latent.parameters(), lr=1e-3)

for epoch in range(30):
    epoch_loss = 0
    for (z_0,) in latent_loader:
        z_0 = z_0.to(device)
        B = z_0.shape[0]
        z_1 = torch.randn_like(z_0)
        t = torch.rand(B, 1, device=device)
        z_t = (1 - t) * z_0 + t * z_1
        target_v = z_1 - z_0
        pred_v = fm_latent(z_t, t)
        loss = F.mse_loss(pred_v, target_v)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/30 | Loss: {epoch_loss/len(latent_loader):.4f}")

fm_latent.eval()
sampled_z = flow_matching_sample(fm_latent, n_samples=16, dim=32, n_steps=50, device=device)
sampled_z = sampled_z * latent_std.to(device) + latent_mean.to(device)
with torch.no_grad():
    fm_images = vae.decode(sampled_z)
show_images(denormalize(fm_images), nrow=8, title='Flow Matching Latent Diffusion Samples')
print("Flow matching: simpler training, comparable results, fewer sampling steps!")

---
## Capstone Exercise: Full Latent Diffusion Pipeline

Bring everything together:

1. **Train a convolutional VAE** on MNIST (latent dim = 32) — or reuse from 9.2
2. **Encode** all training images to latent space
3. **Train a diffusion model** (DDPM or flow matching) on latent codes
4. **Sample**: generate random latent → decode → image
5. **Compare**: pixel-space diffusion vs latent diffusion — quality, speed, training time

**Bonus:**
- Try v-prediction instead of ε-prediction
- Implement offset noise
- Compare DDPM vs flow matching with different step counts

In [ ]:
# Capstone Exercise — YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — Capstone: Full Latent Diffusion Pipeline
torch.manual_seed(42)

print("=" * 60)
print("CAPSTONE: Latent Diffusion vs Pixel-Space Diffusion")
print("=" * 60)

# --- Part 1: Latent diffusion (already trained above) ---
print("\n--- Part 1: Latent Diffusion Samples ---")
latent_model.eval()
vae.eval()

# DDPM samples
ddpm_z = sample_latent_diffusion(latent_model, schedule, n_samples=8, latent_dim=32, device=device)
ddpm_z = ddpm_z * latent_std.to(device) + latent_mean.to(device)
with torch.no_grad():
    ddpm_imgs = vae.decode(ddpm_z)

# Flow matching samples at different step counts
fm_latent.eval()
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
show_row = lambda imgs, row, label: [
    (axes[row, i].imshow(denormalize(imgs[i]).cpu().squeeze(), cmap='gray'),
     axes[row, i].axis('off')) for i in range(min(8, len(imgs)))
] and axes[row, 0].set_ylabel(label, fontsize=11)

show_row(ddpm_imgs, 0, 'DDPM\n(1000 steps)')

for row, n_steps in [(1, 50), (2, 10)]:
    z = flow_matching_sample(fm_latent, n_samples=8, dim=32, n_steps=n_steps, device=device)
    z = z * latent_std.to(device) + latent_mean.to(device)
    with torch.no_grad():
        imgs = vae.decode(z)
    show_row(imgs, row, f'Flow Match\n({n_steps} steps)')

plt.suptitle('Latent Diffusion: DDPM vs Flow Matching', fontsize=14)
plt.tight_layout(); plt.show()

# --- Part 2: Timing comparison ---
print("\n--- Part 2: Speed Comparison ---")
# Pixel-space model
pixel_model = LatentDiffusionMLP(latent_dim=784, hidden_dim=512).to(device)
opt_p = torch.optim.Adam(pixel_model.parameters(), lr=1e-3)

# Time 5 epochs of each
for name, model, opt, loader in [
    ('Pixel (784d)', pixel_model, opt_p, pixel_loader),
]:
    start = time.time()
    for epoch in range(5):
        for (batch,) in loader:
            batch = batch.to(device)
            B = batch.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(batch)
            noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                    sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
            pred = model(noisy, t_idx.float() / T)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
    pixel_time = time.time() - start

latent_model_fresh = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
opt_l = torch.optim.Adam(latent_model_fresh.parameters(), lr=1e-3)
start = time.time()
for epoch in range(5):
    for (batch,) in latent_loader:
        batch = batch.to(device)
        B = batch.shape[0]
        t_idx = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(batch)
        noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
        pred = latent_model_fresh(noisy, t_idx.float() / T)
        loss = F.mse_loss(pred, noise)
        opt_l.zero_grad(); loss.backward(); opt_l.step()
latent_time = time.time() - start

print(f"Pixel-space (784d): {pixel_time:.1f}s for 5 epochs")
print(f"Latent-space (32d): {latent_time:.1f}s for 5 epochs")
print(f"Speedup: {pixel_time/latent_time:.1f}x")
print("\nLatent diffusion: faster training, comparable quality, and scales to high res!")

---
## Summary

This module covered the key techniques that make modern diffusion models practical:

| Topic | Key Takeaway |
|-------|-------------|
| **Resolution problem** | Pixel-space attention is O(n²) in tokens, O(res⁴) for images — infeasible beyond 256×256 |
| **VAE** | Compress images to a low-dimensional latent space with ELBO loss |
| **Latent diffusion** | Do diffusion in latent space — 24-48× smaller, same quality |
| **Stable Diffusion** | VAE + U-Net + text encoder (CLIP/T5) with CFG |
| **v-prediction** | Balanced across all timesteps (vs ε or x₀ prediction) |
| **Noise schedules** | Cosine > linear; offset noise for brightness; zero terminal SNR |
| **DiT** | Transformer replaces U-Net — better scaling, simpler architecture |
| **Flow matching** | Simpler than DDPM: linear interpolation, straight paths, fewer steps |

**The field is moving fast.** The trend is clear: simpler training objectives (flow matching), more scalable architectures (DiT), and better conditioning (larger text encoders). The fundamentals from Modules 1-8 remain the foundation — these advanced techniques build on top of them.